# How much cash remains after full refunds?

Change the ticket size, order count and number of refunds below. Compare six published US domestic online payment paths using the November 2, 2026 snapshot. [Read the underlying Payney comparison and provider terms](https://payney.com/reports/online-card-fees-us-2026-09/?utm_source=github&utm_medium=referral&utm_campaign=refund_notebook#november-recheck).

Assume identical successful USD payments, full refunds and retained original processing fees. Monthly plans, taxes, disputes, FX, shipping and cost of goods are excluded. Shopify Basic requires a paid plan.

In [ ]:
from decimal import Decimal, ROUND_HALF_UP
from urllib.request import urlopen
import json

ticket_usd = Decimal("100.00")
orders = 100
refunds = 10

if not ticket_usd.is_finite() or ticket_usd <= 0 or ticket_usd != ticket_usd.quantize(Decimal("0.01")):
    raise ValueError("Enter a positive dollar amount in whole cents.")
if type(orders) is not int or type(refunds) is not int or not 0 <= refunds <= orders or orders < 1:
    raise ValueError("Enter whole order counts with refunds between zero and total orders.")


In [ ]:
inputs_url = "https://raw.githubusercontent.com/stuseek/payney-payment-fees/657da4308693cc2d27779258f1d3880a88b48410/editions/2026-11-02/recheck-2026-11-02-inputs.json"
with urlopen(inputs_url, timeout=20) as response:
    snapshot = json.load(response)

def calculate(provider, ticket, orders, refunds):
    ticket_cents = int(ticket * 100)
    fee = Decimal(ticket_cents) * Decimal(provider["rate_percent"]) / 100 + provider["fixed_fee_cents"]
    per_order = int(fee.quantize(Decimal("1"), rounding=ROUND_HALF_UP))
    processing = per_order * orders
    return processing, per_order * refunds, ticket_cents * (orders - refunds) - processing

def usd(cents):
    return f"${Decimal(cents) / 100:,.2f}"

print(f"{'Payment path':32} {'Processing':>14} {'Of which refunded':>18} {'Cash remaining':>16}")
for provider in snapshot["providers"]:
    processing, refunded_fees, remaining = calculate(provider, ticket_usd, orders, refunds)
    print(f"{provider['name']:32} {usd(processing):>14} {usd(refunded_fees):>18} {usd(remaining):>16}")


At the default settings, the Stripe row shows $320 in processing fees, including $32 on refunded orders, and $8,680 remaining. All six rows use per-transaction rounding. Compare the individual terms before choosing a provider.